## 3. A grading assistant

Teachers in general have a lot of administrations to do and one of those things is grading. Can we create a simple grade assistant to assist a Swedish teacher in grading? This exercise focuses a lot in prompt engineering and afterwards to postprocess the output using Pydantic.



a) Go into this page with examples of students answers to a particular question. Copy some example texts and paste it into files with names like `student_text_1.txt`, `student_text_2.txt`.



b) Read these data into python and tell your LLM to grade them.



In [34]:
question = "Vad vill du uppnå i framtiden, och varför?"

path = "student_answers/student_text_"

student_answers = []
for i in range(1,6):
    with open(f"{path}{i}.txt", "r", encoding="utf-8") as file:
        student_answers.append(file.read())
        
student_answers

['Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.',
 'I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.',
 'Jag vill utbilda mig till elektriker. Jag tycker om att jobba praktiskt och vill ha ett yrke där man alltid behövs. Det känns också tryggt eftersom man kan få jobb på många olika platser. Dessutom vill jag kunna försörja en framtida familj.',
 'Mitt mål i framtiden är att arbeta som journalist. Jag vill berätta om viktiga saker som händer i världen och hjälpa människor att förstå dem. För att nå dit måste jag studera vidare och öva på att skriva mer varierat. Jag tror också att det är bra att resa och få egna erfarenheter.',
 'I framtiden vill jag bli läkare, framför allt för att jag vill kunna göra en skillnad för människor när de befinner sig i svåra situationer. Yrket lockar mig eftersom det förenar vetenskaplig kunskap med mänskliga möten, vilket jag ser som både en utmaning och

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

prompt = f"""
    Baserat på frågan: {question}
    Betygsätt dessa texter: {student_answers}
"""

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

Här är en bedömning av texterna baserat på frågan "Vad vill du uppnå i framtiden, och varför?":

**Bedömningsskala:**
*   **1/5 – Mycket grundläggande:** Beskriver en yta utan djup eller konkret motivering.
*   **2/5 – Grundläggande:** Specificerar något men med begränsad motivering eller framtidsplanering.
*   **3/5 – Godkänd:** Beskriver ett tydligt mål och ger en rimlig motivering, men kan sakna djup eller bredd.
*   **4/5 – Bra:** Tydligt och specifikt mål med flera välgrundade skäl och/eller insikt i vägen dit.
*   **5/5 – Utmärkt:** Mycket specifikt, djupt reflekterande svar som inkluderar både konkreta mål, starka motiveringar och en förståelse för utmaningar/möjligheter.

---

**1. "Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa."**
*   **Betyg:** 1/5
*   **Motivering:** Målen ("ett jobb", "resa") är väldigt allmänt hållna och saknar specifik inriktning. "Tjäna pengar" är en grundläggande och ganska ytlig motivering, även om den är

c) Prompt to get an output of fields proposed_grade, motivation and improvements.



In [9]:
prompt = f"""
    Baserat på frågan: {question}
    Betygsätt dessa texter mellan F-A ['F', 'E', 'D', 'C' 'B', 'A'], där 'F' är det lägsta betyget, och 'A' är det högsta betyget): {student_answers}
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer, och frågan som texten baseras på i fältet: question
    
    Jag vill ha output i detta format och dessa fält:
    {{
        question: str,
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    INTE markdown.
"""

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

[
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.",
        "proposed_grade": "F",
        "motivation": "Svaret är mycket ytligt och saknar detaljer. Det nämner ett generiskt mål ('ett jobb') och en grundläggande anledning ('tjäna pengar'), samt ett separat önskemål ('resa') utan koppling eller djupare motivation. Det svarar på frågan men med minimal ansträngning.",
        "improvements": "För att nå ett högre betyg behöver texten specificera vilket typ av jobb, varför just det jobbet lockar (utöver pengar), och hur resorna kopplar till framtidsvisionen eller varför de är viktiga. Att visa personlig reflektion och mer detaljerade mål skulle höja betyget."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill oc

d) Now validate this with pydantic model



In [17]:
from pydantic import BaseModel, ValidationError
from typing import Literal, List
import json

class Grade(BaseModel):
    question: str
    student_answer: str
    proposed_grade: Literal['F', 'E', 'D', 'C', 'B', 'A']
    motivation: str
    improvements: str

class GradeListResponse(BaseModel):
    results: List[Grade]

data = json.loads(response.text)

grades = []
for grade in data:
    try:
        grades.append(Grade.model_validate(grade))
    except ValidationError as err:
        print(err)

grade_response = GradeListResponse(results=grades)
print(grade_response.model_dump())

{'results': [{'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.', 'proposed_grade': 'F', 'motivation': "Svaret är mycket ytligt och saknar detaljer. Det nämner ett generiskt mål ('ett jobb') och en grundläggande anledning ('tjäna pengar'), samt ett separat önskemål ('resa') utan koppling eller djupare motivation. Det svarar på frågan men med minimal ansträngning.", 'improvements': 'För att nå ett högre betyg behöver texten specificera vilket typ av jobb, varför just det jobbet lockar (utöver pengar), och hur resorna kopplar till framtidsvisionen eller varför de är viktiga. Att visa personlig reflektion och mer detaljerade mål skulle höja betyget.'}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.', 'proposed_grade': 'D', 'motivation': "

In [32]:
grades

[Grade(question='Vad vill du uppnå i framtiden, och varför?', student_answer='Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.', proposed_grade='F', motivation="Svaret är mycket ytligt och saknar detaljer. Det nämner ett generiskt mål ('ett jobb') och en grundläggande anledning ('tjäna pengar'), samt ett separat önskemål ('resa') utan koppling eller djupare motivation. Det svarar på frågan men med minimal ansträngning.", improvements='För att nå ett högre betyg behöver texten specificera vilket typ av jobb, varför just det jobbet lockar (utöver pengar), och hur resorna kopplar till framtidsvisionen eller varför de är viktiga. Att visa personlig reflektion och mer detaljerade mål skulle höja betyget.'),
 Grade(question='Vad vill du uppnå i framtiden, och varför?', student_answer='I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.', proposed_grade='D', motivation="Svaret är något mer specifik

In [30]:
x = grade_response.model_dump()

In [31]:
x = x["results"]
x

[{'question': 'Vad vill du uppnå i framtiden, och varför?',
  'student_answer': 'Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.',
  'proposed_grade': 'F',
  'motivation': "Svaret är mycket ytligt och saknar detaljer. Det nämner ett generiskt mål ('ett jobb') och en grundläggande anledning ('tjäna pengar'), samt ett separat önskemål ('resa') utan koppling eller djupare motivation. Det svarar på frågan men med minimal ansträngning.",
  'improvements': 'För att nå ett högre betyg behöver texten specificera vilket typ av jobb, varför just det jobbet lockar (utöver pengar), och hur resorna kopplar till framtidsvisionen eller varför de är viktiga. Att visa personlig reflektion och mer detaljerade mål skulle höja betyget.'},
 {'question': 'Vad vill du uppnå i framtiden, och varför?',
  'student_answer': 'I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.',
  'proposed_grade': 'D',
  'motivation'

e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt



In [33]:
folder = "grade_outputs"
os.makedirs(folder, exist_ok=True)

files = {
    "proposed_grade.txt": "proposed_grade",
    "motivation.txt": "motivation", 
    "improvements.txt": "improvements"
    }

for filename, field in files.items():
    with open(f"{folder}/{filename}", "w", encoding="utf-8") as file:
        for grade in grade_response.results:
            value = getattr(grade, field)
            file.write(value + "\n\n")

f) Go [into skolverket for Svenska 1](https://www.skolverket.se/undervisning/gymnasieskolan/program-och-amnen-i-gymnasieskolan/hitta-program-amnen-och-kurser-i-gymnasieskolan-gy11/amne?url=907561864%2Fsyllabuscw%2Fjsp%2Fsubject.htm%3FsubjectCode%3DSVE%26version%3D8%26tos%3Dgy&sv.url=12.5dfee44715d35a5cdfa92a3) and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Paste this into a file called `criterias.txt`.



g) Now repeat b)-e) but with the criterias in your prompt as well. Can you see any differences in the outputs?



In [35]:
question = "Vad vill du uppnå i framtiden, och varför?"

folder_path = "student_answers"

with open(f"{folder_path}/criterias.txt", "r", encoding="utf-8") as file:
    criterias = file.read()

path = f"{folder_path}/student_text_"

student_answers = []
for i in range(1,6):
    with open(f"{path}{i}.txt", "r", encoding="utf-8") as file:
        student_answers.append(file.read())

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

prompt = f"""
    Baserat på dessa betygskriterier: {criterias}
    Betygsätt dessa texter: {student_answers} 
    Texterna baseras på frågan: {question}
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer, och frågan som texten baseras på i fältet: question
    
    Jag vill ha output i detta format och dessa fält:
    {{
        question: str,
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    Ge INTE output i markdown-format.
"""

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

```json
[
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.",
        "proposed_grade": "E",
        "motivation": "Svaret är kortfattat, sammanhängande och begripligt men mycket enkelt i sin formulering och med ytliga motiv. Språket är inte varierat och inga mer komplexa resonemang förs. Eleven förmedlar egna tankar, men dessa är inte fördjupade eller välgrundade.",
        "improvements": "För att nå ett högre betyg skulle eleven behöva utveckla varför pengar är viktigt och hur resor bidrar till ens framtid, med mer konkreta kopplingar. Använd mer varierat ordförråd och mer komplexa meningar för att nyansera tankarna och ge mer välgrundade motiv."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa

In [39]:
stripped_response = response.text.replace("json", "").strip("```")
print(stripped_response)


[
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.",
        "proposed_grade": "E",
        "motivation": "Svaret är kortfattat, sammanhängande och begripligt men mycket enkelt i sin formulering och med ytliga motiv. Språket är inte varierat och inga mer komplexa resonemang förs. Eleven förmedlar egna tankar, men dessa är inte fördjupade eller välgrundade.",
        "improvements": "För att nå ett högre betyg skulle eleven behöva utveckla varför pengar är viktigt och hur resor bidrar till ens framtid, med mer konkreta kopplingar. Använd mer varierat ordförråd och mer komplexa meningar för att nyansera tankarna och ge mer välgrundade motiv."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hun

In [40]:
from pydantic import BaseModel, ValidationError
from typing import Literal, List
import json

class Grade(BaseModel):
    question: str
    student_answer: str
    proposed_grade: Literal['F', 'E', 'D', 'C', 'B', 'A']
    motivation: str
    improvements: str

class GradeListResponse(BaseModel):
    results: List[Grade]

stripped_response = response.text.replace("json", "").strip("```")
data = json.loads(stripped_response)

grades = []
for grade in data:
    try:
        grades.append(Grade.model_validate(grade))
    except ValidationError as err:
        print(err)

grade_response = GradeListResponse(results=grades)
print(grade_response.model_dump())

{'results': [{'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.', 'proposed_grade': 'E', 'motivation': 'Svaret är kortfattat, sammanhängande och begripligt men mycket enkelt i sin formulering och med ytliga motiv. Språket är inte varierat och inga mer komplexa resonemang förs. Eleven förmedlar egna tankar, men dessa är inte fördjupade eller välgrundade.', 'improvements': 'För att nå ett högre betyg skulle eleven behöva utveckla varför pengar är viktigt och hur resor bidrar till ens framtid, med mer konkreta kopplingar. Använd mer varierat ordförråd och mer komplexa meningar för att nyansera tankarna och ge mer välgrundade motiv.'}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.', 'proposed_grade': 'E', 'motivation': "Svaret är kortfatt

In [41]:
folder = "grade_outputs"
os.makedirs(folder, exist_ok=True)

files = {
    "proposed_grade2.txt": "proposed_grade",
    "motivation2.txt": "motivation", 
    "improvements2.txt": "improvements"
    }

for filename, field in files.items():
    with open(f"{folder}/{filename}", "w", encoding="utf-8") as file:
        for grade in grade_response.results:
            value = getattr(grade, field)
            file.write(value + "\n\n")

In [47]:
with open(f"{folder}/full_grading2.txt", "w", encoding="utf-8") as file:
    file.write(f"{grade_response.results[0].question}\n\n\n")
    for i, grading in enumerate(grade_response.results):
        file.write(f"Student {i+1}\nProposed grade: {grading.proposed_grade}\nAnswer: {grading.student_answer}\n\nMotivation: {grading.motivation}\n\nImprovements: {grading.improvements}\n\n\n")

In [ ]:
import pandas as pd

without_criterias = pd.DataFrame([
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.",
        "proposed_grade": "F",
        "motivation": "Svaret är mycket ytligt och saknar detaljer. Det nämner ett generiskt mål ('ett jobb') och en grundläggande anledning ('tjäna pengar'), samt ett separat önskemål ('resa') utan koppling eller djupare motivation. Det svarar på frågan men med minimal ansträngning.",
        "improvements": "För att nå ett högre betyg behöver texten specificera vilket typ av jobb, varför just det jobbet lockar (utöver pengar), och hur resorna kopplar till framtidsvisionen eller varför de är viktiga. Att visa personlig reflektion och mer detaljerade mål skulle höja betyget."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.",
        "proposed_grade": "D",
        "motivation": "Svaret är något mer specifikt än det föregående genom att nämna 'egen lägenhet' och 'en hund'. Motivationen ('bestämma själv') är tydlig för lägenheten. Det är dock fortfarande ganska grundläggande livsmål som inte visar på djupare reflektion kring karriär, bidrag till samhället eller personlig utveckling.",
        "improvements": "För att nå ett högre betyg skulle texten kunna inkludera mer ambitiösa eller utvecklande mål. Hur ska lägenheten uppnås? Finns det några karriärmål som ligger till grund för att kunna uppnå detta? Vad är den djupare motivationen bakom att 'bestämma själv' – vad vill man bestämma över, och varför är det viktigt?"
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Jag vill utbilda mig till elektriker. Jag tycker om att jobba praktiskt och vill ha ett yrke där man alltid behövs. Det känns också tryggt eftersom man kan få jobb på många olika platser. Dessutom vill jag kunna försörja en framtida familj.",
        "proposed_grade": "C",
        "motivation": "Detta svar är konkret med ett tydligt yrkesmål ('elektriker') och flera välgrundade motiv: personligt intresse (praktiskt arbete), jobbsäkerhet (behövs, många jobb) och framtida ansvar (försörja familj). Det visar på en viss framtidsplanering och insikt. Det är ett solidt svar som uppfyller frågans krav väl.",
        "improvements": "För att nå ett högre betyg skulle texten kunna fördjupa sig ytterligare i det personliga intresset för elektrikeryrket, kanske nämna specifika aspekter av yrket som lockar. En touch av ambition utöver trygghet, till exempel att vilja bli specialist inom något område eller bidra med innovation, skulle kunna lyfta det."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "Mitt mål i framtiden är att arbeta som journalist. Jag vill berätta om viktiga saker som händer i världen och hjälpa människor att förstå dem. För att nå dit måste jag studera vidare och öva på att skriva mer varierat. Jag tror också att det är bra att resa och få egna erfarenheter.",
        "proposed_grade": "B",
        "motivation": "Svaret anger ett specifikt och meningsfullt yrkesmål ('journalist') med en tydlig och samhällsinriktad motivation ('berätta om viktiga saker', 'hjälpa människor att förstå'). Det visar också på insikt i vad som krävs för att nå målet (studera, öva på att skriva, resa, erfarenheter), vilket tyder på en genomtänkt plan.",
        "improvements": "För att nå ett A-betyg skulle texten kunna utforska djupare den personliga kopplingen till journalistiken, varför just jag vill berätta, eller vilka typer av berättelser/ämnen som engagerar mest. Att reflektera över utmaningar med yrket eller hur man vill skilja sig åt som journalist skulle kunna ge ytterligare djup."
    },
    {
        "question": "Vad vill du uppnå i framtiden, och varför?",
        "student_answer": "I framtiden vill jag bli läkare, framför allt för att jag vill kunna göra en skillnad för människor när de befinner sig i svåra situationer. Yrket lockar mig eftersom det förenar vetenskaplig kunskap med mänskliga möten, vilket jag ser som både en utmaning och en möjlighet. Jag är medveten om att vägen dit är lång och krävande, men jag ser det som en investering i både min egen utveckling och i samhällets framtid. Dessutom hoppas jag att jag i framtiden kan kombinera mitt arbete med forskning för att bidra till nya medicinska framsteg.",
        "proposed_grade": "A",
        "motivation": "Detta är ett utmärkt svar som inte bara presenterar ett specifikt och ambitiöst mål ('läkare'), utan också en djup och mångfacetterad motivation. Det inkluderar altruistiska skäl ('göra skillnad'), personligt intresse (vetenskap/mänskliga möten), självinsikt om vägens svårigheter, samt en bredare vision (bidra till samhället, forskning). Svaret visar på mognad, engagemang och en väl genomtänkt framtidsplan.",
        "improvements": "Svaret är redan på en mycket hög nivå. För en eventuell ytterligare förfining, även om det knappast behövs för ett A, skulle man kunna nämna specifika områden inom medicinen som intresserar mest, eller reflektera över hur man planerar att hantera de krävande aspekterna av yrket mer konkret."
    }
])

with_criterias = pd.DataFrame([{'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'Jag vill ha ett jobb i framtiden. Det är viktigt så jag kan tjäna pengar. Jag vill också resa.', 'proposed_grade': 'E', 'motivation': 'Svaret är kortfattat, sammanhängande och begripligt men mycket enkelt i sin formulering och med ytliga motiv. Språket är inte varierat och inga mer komplexa resonemang förs. Eleven förmedlar egna tankar, men dessa är inte fördjupade eller välgrundade.', 'improvements': 'För att nå ett högre betyg skulle eleven behöva utveckla varför pengar är viktigt och hur resor bidrar till ens framtid, med mer konkreta kopplingar. Använd mer varierat ordförråd och mer komplexa meningar för att nyansera tankarna och ge mer välgrundade motiv.'}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'I framtiden vill jag bo i en egen lägenhet. Då kan jag bestämma själv. Jag vill också skaffa en hund.', 'proposed_grade': 'E', 'motivation': "Svaret är kortfattat, sammanhängande och begripligt men mycket enkelt i sin formulering och med ytliga motiv. Språket är inte varierat och inga mer komplexa resonemang förs. Elevens tankar är grundläggande och saknar fördjupning av 'varför'.", 'improvements': "För att nå ett högre betyg skulle eleven behöva utveckla vad friheten att 'bestämma själv' innebär och varför det är viktigt för individen. Ge mer detaljerade anledningar till önskningarna och använd ett mer varierat språk med mer komplexa meningsbyggnader för att förtydliga motiven."}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'Jag vill utbilda mig till elektriker. Jag tycker om att jobba praktiskt och vill ha ett yrke där man alltid behövs. Det känns också tryggt eftersom man kan få jobb på många olika platser. Dessutom vill jag kunna försörja en framtida familj.', 'proposed_grade': 'C', 'motivation': 'Texten är sammanhängande, begriplig och har en tydlig disposition där målet presenteras följt av flera välgrundade skäl. Språket är varierat och delvis välformulerat. Elevens tankar är mer utvecklade och motiven mer konkreta och underbyggda än på E-nivå, och det finns en klar anpassning till syfte och mottagare.', 'improvements': 'För att nå ett högre betyg (B/A) skulle eleven kunna nyansera sina tankar ytterligare, till exempel genom att reflektera över utmaningar med yrket, hur yrket utvecklas tekniskt eller hur det kan påverka samhället i stort. Språket skulle kunna bli mer ledigt och innehålla fler goda formuleringar som ger ytterligare djup åt resonemanget.'}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'Mitt mål i framtiden är att arbeta som journalist. Jag vill berätta om viktiga saker som händer i världen och hjälpa människor att förstå dem. För att nå dit måste jag studera vidare och öva på att skriva mer varierat. Jag tror också att det är bra att resa och få egna erfarenheter.', 'proposed_grade': 'C', 'motivation': "Texten är sammanhängande, begriplig och har en tydlig disposition (mål, syfte, väg dit). Motiven är välutvecklade och kopplar det personliga målet till ett samhälleligt syfte ('berätta om viktiga saker som händer i världen och hjälpa människor att förstå dem'). Språket är varierat och delvis välformulerat, och resonemanget är välgrundat.", 'improvements': 'För att nå ett högre betyg (B/A) skulle eleven kunna nyansera reflektionerna kring journalistyrket, till exempel genom att nämna etiska överväganden, vikten av källkritik eller utmaningar med objektivitet. Språket kan utvecklas till att bli mer ledigt och innehålla fler goda formuleringar som ger ytterligare djup åt resonemanget och eventuellt nya, relevanta perspektiv.'}, {'question': 'Vad vill du uppnå i framtiden, och varför?', 'student_answer': 'I framtiden vill jag bli läkare, framför allt för att jag vill kunna göra en skillnad för människor när de befinner sig i svåra situationer. Yrket lockar mig eftersom det förenar vetenskaplig kunskap med mänskliga möten, vilket jag ser som både en utmaning och en möjlighet. Jag är medveten om att vägen dit är lång och krävande, men jag ser det som en investering i både min egen utveckling och i samhällets framtid. Dessutom hoppas jag att jag i framtiden kan kombinera mitt arbete med forskning för att bidra till nya medicinska framsteg.', 'proposed_grade': 'A', 'motivation': 'Texten är utmärkt väldisponerad, sammanhängande och begriplig. Språket är ledigt, varierat och innehåller goda formuleringar. Elevens tankar är välgrundade och nyanserade, kopplar den personliga ambitionen till allmänmänskliga förhållanden (göra skillnad, samhällets framtid) och inkluderar även framåtblickande ambitioner som forskning. Texten visar på ett djup i reflektionen och förmågan att se komplexitet, vilket väl uppfyller kriterierna för A-nivå.', 'improvements': 'Det är svårt att hitta större förbättringsområden då texten uppfyller kriterierna för A-nivå väl. En eventuell ytterligare fördjupning skulle kunna vara att reflektera över globala aspekter av läkaryrket eller specifika etiska dilemman som kan uppstå, men detta skulle gå utöver vad som typiskt förväntas i ett kort svar på en sådan fråga.'}])

In [51]:
without_criterias.to_csv(f"{folder}/grading_without_criterias.csv")
with_criterias.to_csv(f"{folder}/grading_with_criterias.csv")

In [53]:
compare = {
    "grade_without_criterias": without_criterias["proposed_grade"],
    "grade_with_criterias": with_criterias["proposed_grade"],
    "correct_grade": ["E", "E", "C", "C", "A"]
}
df_compare = pd.DataFrame(compare)
df_compare

,grade_without_criterias,grade_with_criterias,correct_grade
0,F,E,E
1,D,E,E
2,C,C,C
3,B,C,C
4,A,A,A


In [54]:
df_compare.to_csv(f"{folder}/comparison.csv")

h) Can you improve the output quality by providing few shot examples?

Yes, by giving clear instructions on what type of output you want, and with examples, there will be less validation errors.

In [55]:
path = "student_answers/student_text"

student_answers = []
for i in range(1,3):
    with open(f"{path}{i}.txt", "r", encoding="utf-8") as file:
        student_answers.append(file.read())
student_answers

['Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bli\nutstöt från gruppen. Let

## utan betygskriterier

In [ ]:
prompt = f"""
    Du är en betygsättnings assistant och ska försöka underlätta för läraren i Svenska 1.
    Betygsätt dessa texter mellan F-A ['F', 'E', 'D', 'C' 'B', 'A'], där 'F' är det lägsta betyget, och 'A' är det högsta betyget): {student_answers}
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer
    
    Jag vill ha output i detta format och dessa fält:
    {{
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    INTE markdown.
"""

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import Literal, List
import json

class Grade(BaseModel):
    student_answer: str
    proposed_grade: Literal['F', 'E', 'D', 'C', 'B', 'A']
    motivation: str
    improvements: str

class GradeListResponse(BaseModel):
    results: List[Grade]

stripped_response = response.text.replace("json", "").strip("```")
data = json.loads(stripped_response)

grades = []
for grade in data:
    try:
        grades.append(Grade.model_validate(grade))
    except ValidationError as err:
        print(err)

grade_response = GradeListResponse(results=grades)
print(grade_response.model_dump())

## med betygskriterier

In [ ]:
with open(f"{folder_path}/criterias.txt", "r", encoding="utf-8") as file:
    criterias = file.read()

prompt = f"""
    Du är en betygsättnings assistant och ska försöka underlätta för läraren i Svenska 1.
    Baserat på dessa betygskriterier: {criterias}
    Betygsätt dessa texter: {student_answers} 
    
    För varje text, lägg till texten som betygsätts i fältet: student_answer
    
    Jag vill ha output i detta format och dessa fält:
    {{
        student_answer: str,
        proposed_grade: char F-A,
        motivation: str 'motivering till betyget',
        improvements: str 'saker som kan förbättras för att nå ett högre betyg
    }}
    
    Output ska vara endast i json-format.
    INTE markdown.
"""

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print(response.text)